# Mini-TP 7 · Seguridad con SAIF — REST (FastAPI)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

Este es **uno de los seis notebooks guiados** del Mini-TP 7. Cada uno asegura **una capa**
de las que construimos en el bimestre. **Elige este si tu modelo se expone por REST (FastAPI)**
(o simplemente para practicar esta estrategia). Con completar **uno** alcanza para la entrega.

### SAIF en esta capa
**Riesgo:** abuso, inyección, DoS y *scraping* para robar el modelo.  
**Controles:** autenticación (API key/JWT) · validación de entrada · *rate limiting*.  
**Elementos SAIF:** bases sólidas · automatizar defensas.

### Qué vas a hacer
1. Ver la **versión vulnerable** y por qué es un riesgo.
2. Aplicar el **control principal** (guiado, ya resuelto como ejemplo).
3. Completar los **`# TODO`** sobre tu propio modelo.
4. Escribir una **reflexión** (amenaza mitigada ↔ elemento de SAIF).

> Corre de punta a punta con **uv**, sin servicios externos.


### Dependencias
```bash
uv add fastapi httpx scikit-learn numpy
```

In [1]:
import numpy as np, time, hashlib, datetime
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

d = load_breast_cancer(); sc = StandardScaler().fit(d.data)
X = sc.transform(d.data); y = d.target
modelo = LogisticRegression(max_iter=5000).fit(X, y)
N = X.shape[1]
print("Modelo listo. Features:", N)

Modelo listo. Features: 30


## 1. Versión VULNERABLE — cualquiera consulta el modelo, sin límites

In [2]:
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient

app_v = FastAPI()
@app_v.post("/predict")
def predict_v(payload: dict):
    arr = np.array(payload["features"]).reshape(1, -1)
    return {"pred": int(modelo.predict(arr)[0])}

cli_v = TestClient(app_v)
# sin credenciales, sin validar, sin límite: todo pasa
print("sin auth:", cli_v.post("/predict", json={"features": X[0].tolist()}).status_code)
# y nada frena 1000 consultas seguidas (scraping para robar el modelo)
print("1000 consultas seguidas: permitidas (riesgo de extracción del modelo)")

sin auth: 200
1000 consultas seguidas: permitidas (riesgo de extracción del modelo)


/Users/belcattaneo/Documents/MIA/MLOPS2/mlops2-minitps/.venv/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 2. Control principal (GUIADO) — auth + validación + rate limiting

In [3]:
from fastapi import Depends, HTTPException
from fastapi.security import APIKeyHeader
from pydantic import BaseModel, conlist

API_KEY = "demo-key-123"          # en producción: desde variable de entorno
app = FastAPI()

api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)
def auth(k: str = Depends(api_key_header)):
    if k != API_KEY:
        raise HTTPException(401, "no autorizado")
    return "cliente"

_hits = {}
def rate_limit(cliente: str = Depends(auth)):
    now = time.time(); w = _hits.setdefault(cliente, [])
    w[:] = [t for t in w if now - t < 60]
    if len(w) >= 5:
        raise HTTPException(429, "demasiadas solicitudes")
    w.append(now); return cliente

class Entrada(BaseModel):
    features: conlist(float, min_length=N, max_length=N)   # tipo, tamaño y número exactos

@app.post("/predict")
def predict(x: Entrada, cliente: str = Depends(rate_limit)):
    arr = np.array(x.features).reshape(1, -1)
    return {"prediccion": int(modelo.predict(arr)[0])}

cli = TestClient(app); h = {"X-API-Key": API_KEY}
print("sin key   :", cli.post("/predict", json={"features": X[0].tolist()}).status_code, "(401)")
print("con key   :", cli.post("/predict", json={"features": X[0].tolist()}, headers=h).status_code, "(200)")
print("mala forma:", cli.post("/predict", json={"features": [0.1, 0.2]}, headers=h).status_code, "(422)")
codes = [cli.post("/predict", json={"features": X[0].tolist()}, headers=h).status_code for _ in range(7)]
print("rate limit:", codes, "(429 al superar 5/min)")

sin key   : 401 (401)
con key   : 200 (200)
mala forma: 422 (422)
rate limit: [200, 200, 200, 429, 429, 429, 429] (429 al superar 5/min)


## 3. Nuestra entrega — los tres `# TODO`, sobre el modelo del repo

Las celdas de arriba usan el modelo de juguete con el que viene el notebook. Esta usa el
nuestro: el `XGBClassifier` de arrestos en Chicago que sirven los otros mini-TPs, con su
contrato de entrada real.

- **TODO 1**: autorización por rol con JWT — token firmado, vencimiento corto, 403 si el rol
  no está autorizado.
- **TODO 2**: registro de auditoría, que es además la medida de gobernanza que pide la consigna.
- **TODO 3**: la entrada es `CrimeReport`, el contrato que comparten los mini-TPs que consumen el
  modelo, que ya trae sus rangos. Validar acá es validar con lo mismo que valida producción.


In [4]:
import datetime
import hashlib
import json
import time

import jwt
from fastapi import Depends, FastAPI, HTTPException
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from fastapi.testclient import TestClient

from arrest_model.model import load_bundle, predict
from arrest_model.schemas import EXAMPLE_REPORT, CrimeReport

# El modelo del repo, el mismo que sirven los otros mini-TPs.
bundle = load_bundle()

JWT_SECRET = "cambiar-en-produccion-con-32-bytes-o-mas"
ROLES_PERMITIDOS = {"analista", "admin"}
AUDITORIA = []


def crear_token(usuario, rol, minutos=15):
    """Firma un token para ese usuario y rol.

    El vencimiento corto es el control: un token es una credencial que viaja, y lo que limita el
    daño de que se filtre es cuánto dura, no cuán largo sea el secreto.
    """
    payload = {
        "sub": usuario,
        "rol": rol,
        "exp": datetime.datetime.now(datetime.UTC) + datetime.timedelta(minutes=minutos),
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")


# TODO 1 ───────────────────────────────────────────────────────────────────────────────
bearer = HTTPBearer(auto_error=False)


def requiere_rol(credencial: HTTPAuthorizationCredentials = Depends(bearer)):
    """Verifica el token y exige un rol autorizado. Devuelve quién es, para auditarlo."""
    if credencial is None:
        raise HTTPException(401, "falta el token")
    try:
        # El algoritmo va declarado: aceptar el que venga adentro del token es la vulnerabilidad
        # clásica de JWT, porque permite presentar uno firmado con `none`.
        claims = jwt.decode(credencial.credentials, JWT_SECRET, algorithms=["HS256"])
    except jwt.PyJWTError as error:
        raise HTTPException(401, f"token inválido: {error}") from None
    if claims.get("rol") not in ROLES_PERMITIDOS:
        raise HTTPException(403, "el rol no puede pedir predicciones")
    return claims["sub"]


# El límite va por usuario del token y no por IP: detrás de un proxy todas las consultas llegan
# con la misma IP, así que limitar por IP castigaría a todos por el abuso de uno.
_consultas = {}


def limite_de_tasa(usuario: str = Depends(requiere_rol)):
    ahora = time.time()
    recientes = [t for t in _consultas.get(usuario, []) if ahora - t < 60]
    if len(recientes) >= 5:
        _consultas[usuario] = recientes
        raise HTTPException(429, "demasiadas solicitudes")
    _consultas[usuario] = [*recientes, ahora]
    return usuario


# TODO 2 ───────────────────────────────────────────────────────────────────────────────
def auditar(usuario, reporte, salida):
    """Deja constancia de quién pidió qué y qué se le respondió.

    Guarda el hash del reporte y no el reporte: un reporte trae la latitud y la longitud de un
    hecho, así que conservarlo entero convertiría el registro de auditoría en una segunda base de
    datos personales. Con el hash alcanza para lo que sirve: probar que dos consultas fueron por
    lo mismo y atar cada respuesta a su entrada.
    """
    canonico = json.dumps(reporte, sort_keys=True, default=str).encode("utf-8")
    AUDITORIA.append(
        {
            "ts": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
            "usuario": usuario,
            "entrada_hash": hashlib.sha256(canonico).hexdigest()[:16],
            "prediccion": salida.arrest,
            "probabilidad": round(salida.probability, 6),
            "model_version": salida.model_version,
        }
    )


# TODO 3 ───────────────────────────────────────────────────────────────────────────────
# La entrada no es una lista de floats: es `CrimeReport`, el contrato que comparten los mini-TPs
# que consumen el modelo. Ya trae lo que valida producción —latitud y longitud dentro de Chicago,
# el tipo primario enumerado, el IUCR con su patrón—, así que validar acá es validar con lo mismo.

app = FastAPI(title="Mini-TP 7 · el modelo de arrestos, asegurado")


@app.post("/predict")
def predecir(reporte: CrimeReport, usuario: str = Depends(limite_de_tasa)):
    salida = predict(bundle, [reporte])[0]
    auditar(usuario, reporte.model_dump(), salida)
    return salida


cli = TestClient(app, raise_server_exceptions=False)
analista = {"Authorization": f"Bearer {crear_token('bel', 'analista')}"}
invitado = {"Authorization": f"Bearer {crear_token('ajeno', 'invitado')}"}
vencido = {"Authorization": f"Bearer {crear_token('bel', 'analista', minutos=-1)}"}

print("sin token       :", cli.post("/predict", json=EXAMPLE_REPORT).status_code, "(401)")
print("token vencido   :", cli.post("/predict", json=EXAMPLE_REPORT, headers=vencido).status_code, "(401)")
print("rol no permitido:", cli.post("/predict", json=EXAMPLE_REPORT, headers=invitado).status_code, "(403)")
fuera = {**EXAMPLE_REPORT, "latitude": 0.0}
print("fuera de Chicago:", cli.post("/predict", json=fuera, headers=analista).status_code, "(422)")
print("analista        :", cli.post("/predict", json=EXAMPLE_REPORT, headers=analista).status_code, "(200)")
codigos = [cli.post("/predict", json=EXAMPLE_REPORT, headers=analista).status_code for _ in range(6)]
print("límite de tasa  :", codigos, "(429 al pasar 5 por minuto)")

print("\nRegistro de auditoría:")
for linea in AUDITORIA:
    print("  ", linea)


sin token       : 401 (401)
token vencido   : 401 (401)
rol no permitido: 403 (403)
fuera de Chicago: 422 (422)
analista        : 200 (200)
límite de tasa  : [200, 200, 200, 429, 429, 429] (429 al pasar 5 por minuto)

Registro de auditoría:
   {'ts': '2026-10-14T17:35:34+00:00', 'usuario': 'bel', 'entrada_hash': 'aa9f5c6a432800db', 'prediccion': 0, 'probabilidad': 0.068433, 'model_version': 1}
   {'ts': '2026-10-14T17:35:34+00:00', 'usuario': 'bel', 'entrada_hash': 'aa9f5c6a432800db', 'prediccion': 0, 'probabilidad': 0.068433, 'model_version': 1}
   {'ts': '2026-10-14T17:35:34+00:00', 'usuario': 'bel', 'entrada_hash': 'aa9f5c6a432800db', 'prediccion': 0, 'probabilidad': 0.068433, 'model_version': 1}
   {'ts': '2026-10-14T17:35:34+00:00', 'usuario': 'bel', 'entrada_hash': 'aa9f5c6a432800db', 'prediccion': 0, 'probabilidad': 0.068433, 'model_version': 1}


## 4. Reflexión

### 1. Qué amenaza frena el límite de tasa, y su relación con el robo del modelo

Frena la extracción del modelo. Un atacante que puede consultar sin límite no necesita robar el artefacto: reconstruye la frontera de decisión preguntando. Con seis campos de entrada y siete features, recorrer una grilla de unos miles de consultas alcanza para entrenar un sustituto que aproxima al original, y con ese sustituto después se buscan entradas adversarias sin tocar el servicio real.

Lo interesante es que acá el límite casi no molesta a nadie legítimo: la plataforma procesa 633 reportes por día, o sea 0,007 por segundo. Un consumidor real jamás se acerca a cinco por minuto. Uno que sí lo hace no está usando el servicio para lo que fue hecho. El límite es barato justamente porque conocemos el volumen.

### 2. Qué elemento de SAIF cubre el control principal

Dos. Bases sólidas, con la autenticación y la autorización: el servicio deja de ser anónimo y cada predicción tiene un responsable. Y automatizar las defensas, con el límite de tasa, que actúa sin que nadie mire: no depende de que alguien note el abuso en un tablero.

El registro de auditoría agrega gobernanza: es lo que permite reconstruir después quién preguntó qué, y lo que una auditoría pide ver.

### 3. Por qué validar la entrada es seguridad y no solo calidad

Porque la entrada sin validar es la puerta por la que pasa todo lo demás, y porque un modelo no avisa cuando le dan algo fuera de su dominio: responde igual, con la misma confianza.

En este modelo se ve concreto. La codificación calcula la distancia a la comisaría más cercana de Chicago con una latitud 0 (que con la longitud de Chicago cae en el Pacífico, frente a Ecuador) esa distancia son miles de kilómetros y el modelo igual devuelve una probabilidad que parece seria. El contrato la rechaza con 422 antes de que el modelo la vea.

Y hay una segunda cara, de disponibilidad: el endpoint de lote del repo acepta como máximo mil reportes por llamada. Sin ese tope, una sola consulta con un millón de reportes voltea el servicio sin necesidad de ningún exploit.

## 5. Cómo se lleva esto al TP integrador

La plataforma del integrador hoy expone su API REST sin ningún control, quien llega al puerto predice y lee los metadatos del modelo. Lo de este notebook se traslada así:

| control | dónde va en la plataforma |
|---|---|
| JWT con rol | una dependencia en `tp1_rest/app.py`, aplicada al router `/v1` |
| límite de tasa | la misma dependencia, encadenada después de la autorización |
| auditoría | el middleware que ya registra cada request, sumándole usuario y hash |
| el secreto | variable de entorno del compose, como el resto de las credenciales |

Tres cosas cambian al pasar del notebook a la plataforma:

 - El estado del límite no sirve con más de una réplica, vive en memoria del proceso, así que con dos réplicas el límite efectivo se duplica. La plataforma corre una sola, pero escalarla pide mover ese estado a un almacén compartido.

 - La auditoría tiene que sobrevivir al contenedor. Una lista en memoria se pierde al reiniciar y un registro de auditoría que se borra solo no es un registro de auditoría. En la plataforma iría al log estructurado que ya emite el servicio, que es lo que persiste.

 - Falta el agujero más grande, ocho de los doce servicios del compose comparten la credencial root de MinIO. Quien la tenga escribe el data lake entero. Ponerle autenticación a la API y dejar esa credencial sería asegurar la puerta y abrir la ventana.

## 6. Qué quedó asegurado

| | antes | después |
|---|---|---|
| sin token | 200, predice | **401** |
| token vencido | 200, predice | **401** |
| rol no autorizado | 200, predice | **403** |
| reporte fuera de Chicago | 200, probabilidad sin sentido | **422** |
| mil consultas seguidas | las mil pasan | **429** después de cinco |
| quién preguntó qué | no queda registro | registro con hash, sin datos personales |
